## GX_05 — The Final Evaluation

**Question:** *How good is our churn model, on customers it has never seen?*

**Notebook Overview**
GX_04 compared models on training and validation customers and froze every decision: the model, its hyperparameters and the classification threshold. This notebook applies that frozen model, unchanged, to the 1501 test customers that no notebook has ever read, and reports the honest, final numbers. Everything here is evaluation, no model choice, no tuning, no threshold change will come from this notebook.

**Decisions (all made in GX_04, on training + validation only)**
| Decision | Choice | Where it was decided |
|---|---|---|
| Model | `MultilayerPerceptronClassifier` — layers [16, 16, 2], maxIter 100 | GX_04 Steps 3–6 (baselines + cross-validation); Step 8 |
| Preparation | GX_02's pipeline steps (indexers → one-hot → assembler → scaler), fitted inside the model pipeline, on training rows only | GX_02 / `src.ml.make_pipeline` |
| Features | 2 categorical (Geography, Gender) + 11 numeric (incl. ZeroBalance, LogBalanceToSalary, Engagement) | GX_02 / `src.ml` |
| Threshold | 0.35 — the best churn-F1 cut on the validation split | GX_04 Step 9 |
| Saved model | the fitted pipeline + its settings, in `/Volumes/workspace/churn/models/` | GX_04 Step 12 |





In [0]:
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

import json

from pyspark.ml import PipelineModel
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

from src.config import GOLD_CHURN, SEED
from src.ml import (
    CATEGORICAL_FEATURES, KEY_COL, LABEL_COL, NUMERIC_FEATURES, SPLIT_COL,
    churn_class_scores, confusion_counts,
)

# --- The frozen decisions, all made in GX_04 on train/validation only -----
# Model: the MLP whose architecture won cross-validation (GX_04 Steps 5 and 8).
# Threshold: 0.35, the best churn-F1 cut on the validation split (GX_04 Step 9).
LAYERS = [16, 16, 2]
MAX_ITER = 100
THRESHOLD = 0.35

# Where GX_04 Step 12 saved the fitted model and its settings.
MODEL_DIR = "/Volumes/workspace/churn/models"
MODEL_PATH = f"{MODEL_DIR}/churn_final_model"
SETTINGS_PATH = f"{MODEL_DIR}/churn_final_model_settings.json"

# Guard: if the model or the threshold is ever reopened in GX_04, set this to
# False here - the evaluation cells then stop BEFORE touching the test set,
# until the decision is frozen again and this flag is set back to True.
FINAL_MODEL_CONFIRMED = True

print("Spark version:", spark.version)

# Same environment setting as GX_04: Spark ML writes temporary files; on serverless
# compute they must live in a Unity Catalog volume.
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.churn.ml_tmp")
os.environ["SPARKML_TEMP_DFS_PATH"] = "/Volumes/workspace/churn/ml_tmp"
print("Spark ML temp path:", os.environ["SPARKML_TEMP_DFS_PATH"])

### 1. Loading the data and verifying the split

**Approach:** Read `gold_churn_features` and separate customers using the fixed `split` column. Before training, verify the split sizes, confirm that no customer appears in multiple sets, check that all required input columns exist, and ensure that labels contain only 0 and 1.

**Why:** The split defined in GX_02 supports consistent training and evaluation throughout the project. These checks detect unexpected changes, such as a rebuilt table or an altered schema, and stop execution before model fitting if a requirement is not met.

In [0]:
gold = spark.table(GOLD_CHURN)

# The three parts of the fixed split (GX_02). Nothing is scored yet.
splits = {name: gold.filter(F.col(SPLIT_COL) == name) for name in ("train", "val", "test")}

# Check 1 - sizes and churn rates per split: the stratified split keeps ~20% churn everywhere.
display(
    gold.groupBy(SPLIT_COL)
    .agg(
        F.count("*").alias("customers"),
        F.sum(LABEL_COL).alias("leavers"),
        F.round(F.avg(LABEL_COL) * 100, 1).alias("churn_rate_pct"),
    )
    .orderBy(SPLIT_COL)
)

# Check 2 - no customer in more than one split.
for a, b in (("train", "val"), ("train", "test"), ("val", "test")):
    overlap = splits[a].join(splits[b], on=KEY_COL).count()
    assert overlap == 0, f"Customers appear in both {a} and {b}: {overlap}"

# Check 3 - every column the pipeline needs is present, and the label is always 0/1 (never missing).
required = [KEY_COL, LABEL_COL, SPLIT_COL] + list(CATEGORICAL_FEATURES) + list(NUMERIC_FEATURES)
missing = [c for c in required if c not in gold.columns]
assert not missing, f"gold table is missing columns: {missing}"
assert gold.filter(F.col(LABEL_COL).isNull() | ~F.col(LABEL_COL).isin(0, 1)).count() == 0, "Found labels other than 0/1 (or missing)"

# Check 4 - the sizes GX_02 fixed (nothing drifted since).
EXPECTED_SIZES = {"train": 7_000, "val": 1_499, "test": 1_501}
for name, expected in EXPECTED_SIZES.items():
    actual = splits[name].count()
    assert actual == expected, f"{name}: {actual} rows, expected {expected}"

print("All checks passed: sizes, no overlap, schema and labels are as GX_02 fixed them.")

### 2. Loading the final pipeline saved in GX_04

**Approach:** Load the exact pipeline fitted and saved in Step 12 of GX_04. It contains GX_02’s preprocessing steps and the selected neural network, fitted on the 7,000 training customers only. Before scoring, verify that its saved settings match the final decisions recorded above.

**Why load rather than refit:** Neural network training may produce slightly different results even with the same seed, particularly if data partitioning changes. Loading the saved pipeline ensures that this notebook evaluates the same fitted model validated in GX_04 and used later for streaming predictions.

**Why not retrain on training and validation data:** Combining these sets would provide 8,499 customers, but would produce a new fitted model. The validation results and the threshold of 0.35 were obtained using the training-only model and would not directly validate the refitted version. We therefore retain the saved model and threshold for this final evaluation.

The pipeline also preserves the fitted preprocessing stages. Test customers are transformed using the scaler and indexers learned exclusively from the training data.

In [0]:
# Load the exact model that left GX_04 (Step 12): preparation steps + the frozen
# network, as ONE fitted pipeline, learned from the 7,000 training customers only.
assert FINAL_MODEL_CONFIRMED, (
    "The final model or threshold is not confirmed. Freeze the decision in GX_04, "
    "then set FINAL_MODEL_CONFIRMED = True in the setup cell before evaluating."
)

final_model = PipelineModel.load(MODEL_PATH)
with open(SETTINGS_PATH) as f:
    saved = json.load(f)

# Guard: the saved model must be the frozen decision - same network, same threshold,
# same inputs. If GX_04 changed and was re-saved, this stops before the test set is read.
network = final_model.stages[-1]
checks = {
    "layers": (list(network.getLayers()), LAYERS),
    "maxIter": (network.getMaxIter(), MAX_ITER),
    "threshold": (saved["threshold"], THRESHOLD),
    "categorical_features": (saved["categorical_features"], list(CATEGORICAL_FEATURES)),
    "numeric_features": (saved["numeric_features"], list(NUMERIC_FEATURES)),
}
for name, (found, frozen) in checks.items():
    assert found == frozen, f"Saved model {name} = {found}, frozen decision = {frozen}"

for i, stage in enumerate(final_model.stages):
    print(f"stage {i}: {stage.__class__.__name__}")
print("Saved model matches the frozen decisions:", ", ".join(checks))


### 3. Predictions on the test set

**Approach:** Score the 1,501 held-out test customers using the saved pipeline. Apply the fixed threshold of 0.35: contact customers whose predicted churn probability is at least 35%.

**Why:** This is the final evaluation on previously unused customers. The Exited column remains unchanged and is used only to calculate evaluation metrics, not to adjust the model or threshold.

In [0]:
# The one opening of the test set: 1,501 customers no notebook has read.
# Exited travels along untouched - from here on it is used for metrics only.
test_pred = (
    final_model.transform(splits["test"])
    .withColumn("p_churn", vector_to_array("probability")[1].cast("double"))
    .withColumn("contact", (F.col("p_churn") >= THRESHOLD).cast("int"))
)

# The bank's contact list: the riskiest customers first.
display(
    test_pred.select(KEY_COL, "p_churn", "contact", LABEL_COL)
    .orderBy(F.desc("p_churn"))
    .limit(20)
)

### 4. Final evaluation results

Approach: Report test-set AUC-PR and AUC-ROC, which assess ranking performance across thresholds. At the fixed threshold of 0.35, report accuracy, precision, recall, and F1 for the churn class, alongside a labelled confusion matrix.

Display validation metrics beside the test results. Both sets are scored by the same fitted model, so the comparison assesses performance on different held-out customers without changes to training or model settings.

Why the churn proportion matters: Approximately 20% of test customers churn, giving a random ranking an expected AUC-PR of around 0.20. Performance above this baseline indicates predictive value, although small differences may reflect sampling variation.

In [0]:
# --- Threshold-free quality (judges the whole ranking) --------------------
auc_pr_eval = BinaryClassificationEvaluator(
    labelCol=LABEL_COL, rawPredictionCol="probability", metricName="areaUnderPR"
)
auc_roc_eval = BinaryClassificationEvaluator(
    labelCol=LABEL_COL, rawPredictionCol="probability", metricName="areaUnderROC"
)


def final_metrics(preds):
    """Metrics of one prediction DataFrame at the frozen threshold.

    The confusion counts and the churn-class scores come from src/ml.py (the same
    definitions GX_04 used), applied to the `contact` decision instead of Spark's
    default 0.5 cut.
    """
    counts = confusion_counts(preds, decision_col="contact")
    return {
        "auc_pr": round(auc_pr_eval.evaluate(preds), 3),
        "auc_roc": round(auc_roc_eval.evaluate(preds), 3),
        **churn_class_scores(counts, digits=3),
        "confusion": counts,
    }


# The same fitted model on validation - the comparison below is about
# generalisation only, never about two different models.
val_pred = (
    final_model.transform(splits["val"])
    .withColumn("p_churn", vector_to_array("probability")[1].cast("double"))
    .withColumn("contact", (F.col("p_churn") >= THRESHOLD).cast("int"))
)
test_metrics = final_metrics(test_pred)
val_metrics = final_metrics(val_pred)

display(
    spark.createDataFrame(
        [
            (s, m["auc_pr"], m["auc_roc"], m["accuracy"], m["precision_churn"], m["recall_churn"], m["f1_churn"])
            for s, m in (("validation", val_metrics), ("test", test_metrics))
        ],
        ["split", "auc_pr", "auc_roc", "accuracy", "precision_churn", "recall_churn", "f1_churn"],
    )
)

c = test_metrics["confusion"]
n_test = sum(c.values())
n_leavers = c["TP"] + c["FN"]
print(f"Test churn: {n_leavers}/{n_test} = {n_leavers / n_test:.1%} - the random-ranking AUC-PR floor.")
print(f"At threshold {THRESHOLD}: contacted {c['TP'] + c['FP']} of {n_test} customers and caught {c['TP']} of {n_leavers} leavers.")

display(
    spark.createDataFrame(
        [
            (f"contacted (p_churn >= {THRESHOLD})", c["TP"], c["FP"]),
            ("not contacted", c["FN"], c["TN"]),
        ],
        ["decision", "actually CHURNED", "actually STAYED"],
    )
)